## Generating RAG Answers

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

In [12]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [14]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join the course if you just discovered it recently.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.'

In [18]:
assistant.total_cost()

0.000567

In [23]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [25]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id
}

rag_result

{'question': 'Can I still join the course if I only found it recently?',
 'answer_llm': 'Yes — you can still join the course if you just discovered it recently.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [26]:
rec

{'question': 'Can I still join the course if I only found it recently?',
 'document': '74eb249bbf'}

In [27]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id
    }
    
    return result

In [28]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I only found it recently?',
 'answer_llm': 'Yes, you can still join if you just discovered the course. If you want a certificate, though, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [29]:
assistant.reset_usage()

In [32]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [34]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/425 [00:00<?, ?it/s]

In [35]:
results[:2]

[{'question': 'Can I still join the course if I only found it recently?',
  'answer_llm': 'Yes, you can still join the course if you just found it recently.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now, or can I still participate?',
  'answer_llm': 'Yes, you can still participate — you can start the course whenever you want. The videos and GitHub materials are available, and you can begin learning and submitting homework while the submission form is open.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submiss

In [37]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [38]:
assistant.total_cost()

0.37971300000000013

In [39]:
df_answers = pd.DataFrame(answers)
df_answers.head()

,question,answer_llm,answer_orig,document
0,Can I still join the course if I only found it...,"Yes, you can still join the course if you just...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"Is it too late to start the course now, or can...","Yes, you can still participate — you can start...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join late, will I still be able to get a ...","Yes, but only if you finish with the live coho...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to qualify for the certif...,Yes — you can still qualify for the certificat...,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,Are project submissions still open for new stu...,"Yes — new students can still join, but if you ...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [40]:
df_answers.to_csv("data/rag-answers-new.csv", index=False)

## LLM as a Judge

In [41]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [44]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [45]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [46]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [47]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [49]:
rec = answers[0]

In [50]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
Can I still join the course if I only found it recently?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join the course if you just found it recently.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [51]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joiners may still participate, but certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.', score='good')

In [52]:
calc_price(usage)

{'input_cost': 0.00022574999999999998,
 'output_cost': 0.0002385,
 'total_cost': 0.00046425}

In [53]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [54]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says the student can still join, and that certificate eligibility depends on submitting the project while submissions are still being accepted. This preserves the key meaning without distortion.', score='good')

In [56]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [57]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/425 [00:00<?, ?it/s]

In [58]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [59]:
df_eval = pd.DataFrame(evaluations)
df_eval.head(3)

,question,document,score,reasoning
0,Can I still join the course if I only found it...,74eb249bbf,good,The AI answer preserves the key information fr...
1,"Is it too late to start the course now, or can...",74eb249bbf,good,The AI answer matches the ground truth: it say...
2,"If I join late, will I still be able to get a ...",74eb249bbf,good,The AI answer preserves the core point that la...


In [63]:
calc_total_price(usages)

0.27664575

In [65]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 404/425 = 95.06%


In [68]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
7,"If I didn’t register for LLM Zoomcamp, can I s...",977bf7786c,bad,The AI answer fails to convey the ground truth...
28,Do I need to review other students’ capstone p...,69d122f12e,bad,The AI answer is incorrect. The ground truth s...
34,Can I still earn the certificate even if I mis...,9f689c185f,bad,The AI answer captures the key point that home...
41,Do you know what the upcoming offering date is...,bd31146b0e,bad,The ground truth gives a specific upcoming off...
42,When should I expect the next session of the c...,bd31146b0e,bad,The ground truth gives a specific start time: ...


In [75]:
# index of bad answers according to eval
df_eval[df_eval["score"] == "bad"].index

Index([  7,  28,  34,  41,  42,  83,  84, 199, 223, 306, 307, 317, 321, 356,
       359, 369, 401, 403, 412, 414, 421],
      dtype='int64')

In [77]:
# for index 7, the llm didn't know the answer 
answers[7]

{'question': 'If I didn’t register for LLM Zoomcamp, can I still follow along and turn in assignments?',
 'answer_llm': "I don't know.",
 'answer_orig': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'document': '977bf7786c'}

In [81]:
# which was due to the poor data returned by the search function (used text search)
assistant.search(answers[7]['question'])

[{'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Where can I find previous LLM Zoomcamp projects?',
  'answer': 'You can browse previous LLM Zoomcamp project submissions here:\n\n- [2024 projects](https://courses.datatalks.club/llm-zoomcamp-2024/projects)\n- [2025 projects](https://courses.datatalks.club/llm-zoomcamp-2025/projects)\n\nThese pages show submitted repositories and can help you understand the expected scope and quality of capstone projects.',
  'doc_id': '930286278d'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The video

In [82]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

## Agent Evaluation

In [85]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [86]:
ground_truth[:2]

[{'question': 'Can I still join the course if I only found it recently?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now, or can I still participate?',
  'document': '74eb249bbf'}]

In [87]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [88]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

In [91]:
doc_idx

{'74eb249bbf': {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 '977bf7786c': {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'},
 '489dd1c9d9': {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/work

In [117]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [118]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [119]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [120]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [121]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Can I still join the course if I only found it recently?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"join course late found recently enrollment still join course after course started"}', call_id='call_pSqgBZSMBaf6C7LhSZqaDViC', name='search', type='function_call', id='fc_08860cceb7e848dd006a43521552b4819b8c441b1c4fb57b3c', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_pSqgBZSMBaf6C7LhSZqaDViC',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re s

In [122]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [123]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"join course late found recently enrollment still join course after course started"}'}]

In [124]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [125]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Can I still join the course if I only found it recently?',
 'answer_agent': 'Yes — you can still join the course even if you discovered it recently.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"join course late found recently enrollment still join course after course started"}'}],
 'cost': Decimal('0.0010515'),
 'document': '74eb249bbf'}

In [190]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [191]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [199]:
import json

df_agent = pd.DataFrame(agent_answers)
# fixes json.loads object error later due to invalid json quotes used (single quotes instead of double quotes)
df_agent["tool_calls"] = df_agent["tool_calls"].apply(json.dumps)
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,Can I still join the course if I only found it...,Yes — you can still join the course even if yo...,"Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.00094275,74eb249bbf
1,"Is it too late to start the course now, or can...",Yes — you can still start and participate now....,"Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.00119025,74eb249bbf
2,"If I join late, will I still be able to get a ...","Yes, but only if you join while the course is ...","Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.00107775,74eb249bbf
3,What do I need to do to qualify for the certif...,"If you’re joining now, you can still qualify f...","Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.0013875,74eb249bbf
4,Are project submissions still open for new stu...,"Yes — new students can still join the course, ...","Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.00098325,74eb249bbf


In [200]:
df_agent["cost"].sum()

Decimal('0.06224625')

In [201]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [209]:
df_agent = pd.read_csv("data/agent-answers.csv")
# df_agent["tool_calls"] = df_agent["tool_calls"].apply(json.dumps)
agent_answers = df_agent2.to_dict(orient="records")

In [210]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [211]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [212]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [213]:
json.loads(agent_answers2[0]["tool_calls"])

[{'name': 'search',
  'arguments': '{"query":"join the course found it recently enrollment late join can I still join"}'}]

In [214]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It correctly says the user can still join the course if they found it recently, and it also includes the condition for receiving a certificate: submitting the project while submissions are still open.', answer_score='good', trajectory_reasoning='The search query was relevant to the question and included key ideas like joining the course and finding it recently. Only one search call was made, which is reasonable, and it supported the final answer.', trajectory_score='good')

In [182]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [183]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [184]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [185]:
df_agent_eval = pd.DataFrame(agent_evaluations)
df_agent_eval.head()

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,Can I still join the course if I only found it...,74eb249bbf,good,The agent's answer matches the ground truth: i...,good,The single search query was relevant to the qu...
1,"Is it too late to start the course now, or can...",74eb249bbf,good,The agent’s answer matches the ground truth. I...,good,The search query was relevant to the question ...
2,"If I join late, will I still be able to get a ...",74eb249bbf,good,The agent’s answer matches the ground truth on...,good,The search query is broadly relevant to the qu...
3,What do I need to do to qualify for the certif...,74eb249bbf,good,The agent answer includes the key idea from th...,good,The tool query was relevant to the question be...
4,Are project submissions still open for new stu...,74eb249bbf,good,The agent answer matches the ground truth: it ...,good,The single search query is relevant and includ...


In [186]:
calc_total_price(usages)

0.053671500000000004

In [187]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    48
bad      2
Name: count, dtype: int64

In [188]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [189]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)